# Data Cleaning Pipeline
Takes in a run name (see notebook 0), cleans off the spinup time and removes values that we do not want the neural network to see. Aggregates into an array. Segmented so the next notebook doesn't change much

In [2]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import re
import time

In [1]:
from helpers.defs import read_mod_file #need to restart kernel to get these updated!!
from helpers.defs import check_nans

In [2]:
## Config block, change things here
RUN_NAME = "MDF_1"
forcing_loc_list = ["centralarctic", "coastalcanada", "sibchuk", "barents"]
saved_vars = ['evap','fsnow','frazil','fswabs','flw','flwout','fsens','fsurf','flat','frain','Tair','Qa','fsw','fcondtop','meltt','meltb','meltl','snoice','dsnow','congel','sst','sss','Tf','fhocn','melts', 'aicen','vicen', 'vsnon']
saved_tracers_dict = {"Tsfcn":[1],"qice":[2,3,4,5,6,7,8,9],"qsno": [10, 11, 12], "sice":[13,14,15,16,17,18,19,20]} #using fortran/.sel indexing, check log file for tracer defs
changed_param = ["ksno"]
changed_parameterizations = ["ktherm"]
start_year = 1975
spinup_years=5
agg_hours = 24

In [3]:
SCRATCH_DIR = Path(f"/scratch/alpine/dagr5998/{RUN_NAME}")
PARAM_DIR = Path(f"/projects/dagr5998/icepack_ML/parameters/{RUN_NAME}")  
OUT_DIR = Path(f"/projects/dagr5998/cleaned_data/01icepack_array_output/{RUN_NAME}_daily")

In [4]:
#modfile = SCRATCH_DIR/modfile_name / f"{modfile_name}.mods"
#ENSEMBLE_DIR = SCRATCH_DIR / modfile_name

In [5]:
OUT_DIR.mkdir(parents=True)

In [7]:
def clean_run(path, saved_vars, saved_tracers, spinup_years=2):
    # Takes the path to the dataset and basic inputs and retearns cleaned dataset
    with xr.open_dataset(path, decode_times=False) as ds:
        ds = ds[saved_vars + ["trcrn"]].load()
    ds = ds.sel(ni=3).drop_vars("ni")

    out = {v: ds[v] for v in saved_vars}
    for name, idxs in saved_tracers.items():
        for i in idxs:
            out[f"{name}{i}"] = ds["trcrn"].sel(ntrcr=i).drop_vars("ntrcr")
    return xr.Dataset(out).isel(time=slice(spinup_years * 365 * 24, None))

In [8]:
def aggregate(ds, hours=24):
    #cuts down on memory, keeps trend
    if hours == 1:
        return ds
    return ds.coarsen(time=hours, boundary="trim").mean()

In [10]:
def to_array(ds, saved_vars, saved_tracers_dict):
    #Takes in a cleaned xarray netcdf dataset and the pulled variables and puts it into (N, T, C): (Number of runs, Timesteps, Channel (variables)) 
    names = list(saved_vars) + [f"{n}{i}" for n, index in saved_tracers_dict.items() for i in index]
    #print(names)
    ncat = ds.sizes["ncat"]
    cols, channels = [], []
    for i in names:
        ds_i = ds[i]
        if "ncat" in ds_i.dims:
            cols.append(ds_i.transpose("time", "ncat").values)
            channels += [f"{i}_ncat{c}" for c in ds_i["ncat"].values]
        else:
            cols.append(ds_i.values[:,None])
            channels += [i]
    return np.concatenate(cols, axis=1).astype(np.float32), channels

In [11]:
run_dirs = sorted(SCRATCH_DIR.glob("*_set_*"))
#run_dirs = sorted(SCRATCH_DIR.glob("barents_set_*"))
parameterizations=[]
X_list, Y_list, sets, locations = [], [], [], []
channels = None
t0 = time.time()
for run_dir in run_dirs:
    loc, set_name = run_dir.name.split("_", 1)
    params = read_mod_file(PARAM_DIR / f"{set_name}.mods")

    ds = next((run_dir / "history").glob("*.nc"))
    ds_i = aggregate(clean_run(ds, saved_vars, saved_tracers_dict,  spinup_years), agg_hours)
    X_i, channels_i = to_array(ds_i, saved_vars,saved_tracers_dict)   # one run

    if channels is None:
        channels = channels_i
    assert channels_i == channels 

    X_list.append(X_i)
    Y_list.append([params[k] for k in changed_param])     
    sets.append(int(set_name.rsplit("_", 1)[1]))
    locations.append(loc)
    parameterizations.append([params[k] for k in changed_parameterizations])
    print(f"{time.time() - t0:.0f}s elapsed")
X = np.stack(X_list) #(N, T, C) time series for every run
Y = np.array(Y_list, dtype=np.float32) #(N, P) parameter values for each run
sets = np.array(sets, dtype=np.int32)  #(N,) this is param/parameterization it was run on
locations = np.array(locations) #(N,) forcing loc

print(X.shape, Y.shape, sets.shape, locations.shape)

58s elapsed
108s elapsed
206s elapsed
324s elapsed
427s elapsed
473s elapsed
566s elapsed
707s elapsed
836s elapsed
970s elapsed
1170s elapsed
1392s elapsed
1482s elapsed
1616s elapsed
1771s elapsed
1900s elapsed
2118s elapsed
2342s elapsed
2469s elapsed
2693s elapsed
2861s elapsed
3004s elapsed
3225s elapsed
3430s elapsed
3672s elapsed
3847s elapsed
4121s elapsed
4371s elapsed
4537s elapsed
4762s elapsed
4940s elapsed
5123s elapsed
(32, 16060, 140) (32, 1) (32,) (32,)


In [12]:
channels.index("aicen_ncat1")

25

In [13]:
np.save(OUT_DIR / "X.npy", X)
np.save(OUT_DIR/ "Y.npy", Y)
np.save(OUT_DIR / "sets.npy", sets)
np.save(OUT_DIR/ "locations.npy", locations)
np.save(OUT_DIR / "parameterizations.npy", np.array(parameterizations))
config = {
    "channels":     channels,
    "parameters": changed_param,
    "parameterizations":  changed_parameterizations,
    "shape":        list(X.shape),
    "start_year":         start_year,
    "first_year":         start_year + spinup_years,
    "source_dir":   str(SCRATCH_DIR),
    "saved_vars":    saved_vars,
    "saved_tracers": saved_tracers_dict,
    "aggregated hours": agg_hours,
    "created":       datetime.now().isoformat(timespec="seconds"),
    "notes": {
        "tracer_naming": "Tracer channel numbers are Fortran ntrcr indices from the run log, not layer numbers. qice2-qice9 are ice layers 1-8 are so forth ",
    },
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2)) #creates a record to go along with the output


3707